In [20]:
import os

In [21]:
%pwd

'c:\\Users\\aabhi\\Desktop\\kidney_disease\\Kidney-Disease-Classification'

In [22]:
os.chdir("../")

In [23]:
%pwd

'c:\\Users\\aabhi\\Desktop\\kidney_disease'

In [24]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int


class ConfigurationManager:

    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(
        self
    ) -> PrepareBaseModelConfig:

        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        prepare_base_model_config = (
            PrepareBaseModelConfig(
                root_dir=config.root_dir,
                base_model_path=config.base_model_path,
                updated_base_model_path=config.updated_base_model_path,
                params_image_size=self.params.IMAGE_SIZE,
                params_learning_rate=self.params.LEARNING_RATE,
                params_include_top=self.params.INCLUDE_TOP,
                params_weights=self.params.WEIGHTS,
                params_classes=self.params.CLASSES
            )
        )

        return prepare_base_model_config

In [25]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [26]:
import tensorflow as tf
from pathlib import Path


class PrepareBaseModel:

    def __init__(self, config):
        self.config = config

    def get_base_model(self):

        self.model = tf.keras.applications.MobileNetV2(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top,
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):

        # Freeze all layers
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        # Freeze layers till specific layer
        elif (freeze_till is not None) and (freeze_till > 0):

            for layer in model.layers[:freeze_till]:
                layer.trainable = False

        # Add custom classifier
        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()

        return full_model

    def update_base_model(self):

        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

    @staticmethod
    def save_model(path: Path, model):

        path = Path(path)

        # create parent directories
        path.parent.mkdir(parents=True, exist_ok=True)

        model.save(str(path))
    

In [27]:
import os 
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [28]:
import tensorflow as tf
from pathlib import Path


class PrepareBaseModel:

    def __init__(self, config):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.applications.MobileNetV2(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top,
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):

        # Freeze all layers
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False

        # Freeze layers till specific layer
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:freeze_till]:
                layer.trainable = False

        # Custom classifier
        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()

        return full_model

    def update_base_model(self):

        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )

    @staticmethod
    def save_model(path: Path, model):
        path = Path(path)

        # create parent directory if needed to avoid error
        path.parent.mkdir(parents=True, exist_ok=True)

        model.save(str(path))

In [29]:
try:

    config = ConfigurationManager()

    prepare_base_model_config = (
        config.get_prepare_base_model_config()
    )

    prepare_base_model = PrepareBaseModel(
        config=prepare_base_model_config
    )

    prepare_base_model.get_base_model()

    prepare_base_model.update_base_model()

except Exception as e:
    raise e

FileNotFoundError: [Errno 2] No such file or directory: 'src\\cnnClassifier\\config\\config.yaml'